# Chapter 11: Transformers and Large Language Models
**Part III — Deep Learning**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

Transformer architecture, self-attention, BERT vs GPT, LoRA, and RLHF.

## Learning Objectives
See Chapter 11 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Tokenisation with Hugging Face


In [ ]:
# Tokenisation with Hugging Face
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Artificial intelligence is transforming the world."
tokens = tokenizer(text, return_tensors="pt",
                  padding=True, truncation=True, max_length=512)

print("Token IDs:", tokens["input_ids"])
print("Decoded:", tokenizer.decode(tokens["input_ids"][0]))

# Tokenise a batch
batch = ["The cat sat on the mat.", "Deep learning is powerful."]
batch_tokens = tokenizer(batch, padding=True, truncation=True,
                        return_tensors="pt")

## Load and fine-tune BERT on SST-2 sentiment analysis


In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer
from transformers import TrainingArguments
import numpy as np
from datasets import load_dataset

# Load and fine-tune BERT on SST-2 sentiment analysis
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised_train,
    eval_dataset=tokenised_val,
)
trainer.train()

## Visualising self-attention weights


In [ ]:
# Visualising self-attention weights
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Simulate a small attention computation
torch.manual_seed(42)
seq_len, d_model = 8, 16
tokens = ["The", "cat", "sat", "on", "the", "mat", ".", "[EOS]"]

Q = torch.randn(1, seq_len, d_model)
K = torch.randn(1, seq_len, d_model)
V = torch.randn(1, seq_len, d_model)

# Scaled dot-product attention
d_k = d_model
scores = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)
attn_weights = F.softmax(scores, dim=-1)   # (1, seq, seq)

attn = attn_weights[0].detach().numpy()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(attn, cmap='Blues', vmin=0, vmax=attn.max())
ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens, rotation=45, ha='right')
ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
ax.set_xlabel("Key (attended to)", fontsize=12)
ax.set_ylabel("Query (attending from)", fontsize=12)
ax.set_title("Self-Attention Weights\n(brighter = stronger attention)", fontsize=12, fontweight='bold')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{attn[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if attn[i,j] > 0.15 else '#1F3864')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig('ch11_attention.png', dpi=150, bbox_inches='tight')
plt.show()

## LoRA parameter count comparison


In [ ]:
# LoRA parameter count comparison
def count_lora_params(d, k, r):
    """Compare full fine-tuning vs LoRA parameter counts."""
    full_params = d * k
    lora_params = r * (d + k)
    reduction   = 1 - lora_params / full_params
    return full_params, lora_params, reduction

print("LoRA Parameter Reduction Analysis")
print("=" * 60)
print(f"{'Layer':<30} {'Full FT':>10} {'LoRA (r=8)':>12} {'Reduction':>10}")
print("-" * 60)

layers = [
    ("Attention Q (768×768)", 768, 768),
    ("Attention Q (1024×1024)", 1024, 1024),
    ("FFN layer (768×3072)", 768, 3072),
    ("FFN layer (1024×4096)", 1024, 4096),
    ("GPT-3 layer (12288×12288)", 12288, 12288),
]

for name, d, k in layers:
    fp, lp, red = count_lora_params(d, k, r=8)
    print(f"{name:<30} {fp:>10,} {lp:>12,} {red:>9.1%}")

print()
print("LoRA insight: for a 768×768 attention matrix, rank-8 LoRA")
print(f"trains only {count_lora_params(768,768,8)[1]:,} parameters instead of {count_lora_params(768,768,8)[0]:,}.")

---

## 📚 Review Questions

See Chapter 11 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 11 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*